# CS781 Phase 5 - final analysis, ensembles and submission files (CPU only)
Reads the four run bundles (`bundle_p5_1.zip` ... `bundle_p5_4.zip`; zipped or already unzipped by Kaggle) and builds
every table for the report, the ensembles and the submission files. Nothing is trained on dev or test: ensembles
average saved probabilities; members and thresholds are chosen on dev (dev scores of ensembles and per-statute
thresholds are therefore optimistic - the cross-fitted numbers and the leaderboard are the honest checks).

Tables: all runs; technique x model; study (B and D); cost; decision rules (0.5 / global / per-statute); rare vs
frequent statutes; most-missed and most-over-predicted statutes and confused pairs; co-occurrence; where the attention
goes; ensembles; then the submission files in order, each a different method.

**Inputs:** the four bundles + `cs781-p5-code`. Internet ON, `HF_TOKEN`, **Accelerator: None**. Save & Run All (~5 min).

In [1]:
import os, sys, json, re, glob, shutil, time, zipfile
import numpy as np
W = "/kaggle/working"
WORK, P5DATA, RUNS = f"{W}/p5code", "/tmp/p5data", "/tmp/p5runs"
for d in (RUNS, P5DATA):
    os.makedirs(d, exist_ok=True)
def find_dir(name, root="/kaggle/input"):
    hits = sorted(glob.glob(os.path.join(root, "**", name), recursive=True), key=len)
    assert hits, f"{name} not found under {root}"
    return os.path.dirname(hits[0])
shutil.rmtree(WORK, ignore_errors=True)
shutil.copytree(find_dir("lsi_train.py"), WORK)
os.chdir(WORK); sys.path.insert(0, WORK)
import lsi_data as D, lsi_eval as E
from huggingface_hub import hf_hub_download, list_repo_files
from kaggle_secrets import UserSecretsClient
REPO = "Exploration-Lab/CS-781-Capstone"
tok_ = UserSecretsClient().get_secret("HF_TOKEN")
for f in [f for f in list_repo_files(REPO, repo_type="dataset", token=tok_) if f.startswith("Phase_5")]:
    dst = os.path.join(P5DATA, f); os.makedirs(os.path.dirname(dst), exist_ok=True)
    shutil.copy(hf_hub_download(REPO, f, repo_type="dataset", token=tok_), dst)
EXPS = {e["tag"]: e for e in json.load(open(f"{WORK}/experiments_p5.json"))}
for z in glob.glob("/kaggle/input/**/bundle_p5_*.zip", recursive=True):
    zipfile.ZipFile(z).extractall(RUNS)
for p in glob.glob("/kaggle/input/**/summary.json", recursive=True):       # bundles Kaggle already unzipped
    d = os.path.dirname(p); t = os.path.basename(d)
    if t in EXPS and not os.path.exists(f"{RUNS}/{t}"):
        shutil.copytree(d, f"{RUNS}/{t}")
FOUND = sorted(t for t in os.listdir(RUNS) if t in EXPS and os.path.exists(f"{RUNS}/{t}/dev_prob.npz"))
print(len(FOUND), "runs found; missing:", " ".join(sorted(set(EXPS) - set(FOUND))) or "none")

Phase_5_LSI/Test/test.json: reconstructing file:   0%|          |  0.00B / 98.1MB            

Phase_5_LSI/Test/test.json: downloading bytes:           |  0.00B            

Phase_5_LSI/Train/dev.json: reconstructing file:   0%|          |  0.00B / 79.4MB            

Phase_5_LSI/Train/dev.json: downloading bytes:           |  0.00B            

Phase_5_LSI/Train/train.json: reconstructing file:   0%|          |  0.00B /  299MB            

Phase_5_LSI/Train/train.json: downloading bytes:           |  0.00B            

62 runs found; missing: none


## Tables 1-5

In [2]:
S = {t: json.load(open(f"{RUNS}/{t}/summary.json")) for t in FOUND}
def npz(t, split):
    p = f"{RUNS}/{t}/{split}_prob.npz"
    if not os.path.exists(p):
        return None
    z = dict(np.load(p)); z["P"] = z["P"].astype(np.float32)
    return z
DEV = {t: npz(t, "dev") for t in FOUND}
TEST = {t: npz(t, "test") for t in FOUND}
ref_ids = [str(i) for i in DEV[FOUND[0]]["ids"]]
Y = DEV[FOUND[0]]["Y"].astype(np.uint8)
for t in FOUND:
    assert [str(i) for i in DEV[t]["ids"]] == ref_ids, f"{t}: dev order differs"
HAVE_TEST = all(TEST[t] is not None for t in FOUND)
test_ids = [str(i) for i in TEST[FOUND[0]]["ids"]] if HAVE_TEST else []
info = S[FOUND[0]]["cache_info"]
K = Y.shape[1]
# training frequency: from a full-data run (study runs saw a subset)
freq = np.array(next(S[t]["train_label_freq"] for t in FOUND if EXPS[t]["group"] != "S"))
f1 = lambda t: S[t]["dev"]["global"]["macro_f1"]
cf = lambda t: S[t]["dev"]["crossfit"]["per_label"]
MODELS = "ABCDE"
TECHS = ["lorma", "adalora", "qlora", "adapter", "prefix", "partial", "full", "prompt"]
RUN = {(EXPS[t]["model"].upper(), t.split("_", 1)[1]): t for t in FOUND}
L = ["=== REPORT phase 5 ===", f"data: {json.dumps(info)}",
     f"{len(FOUND)} runs; {K} statutes; dev {len(Y)} documents ({Y.sum(1).mean():.2f} statutes each); test {len(test_ids)}", "",
     "ALL RUNS by dev macro-F1 at the best global threshold | threshold | per-statute thresholds: in-sample (cross-fitted) | "
     "at 0.5 | micro-F1 | subset acc | statutes/doc | trainable M | train min | peak GB | ms/doc"]
for t in sorted(FOUND, key=f1, reverse=True):
    s, d = S[t], S[t]["dev"]
    L.append(f"{t:22s} {d['global']['macro_f1']:.4f} | {d['global_threshold']:.2f} | {d['per_label']['macro_f1']:.4f} ({cf(t):.4f}) | "
             f"{d['at_0.5']['macro_f1']:.4f} | {d['global']['micro_f1']:.4f} | {d['global']['subset_acc']:.4f} | "
             f"{d['global']['pred_per_doc']:.2f} | {s['params_trainable'] / 1e6:7.2f} | {s['train_minutes']:5.1f} | "
             f"{s.get('peak_mem_gb') or 0:4.1f} | {s['infer_ms_per_doc']:6.1f}")
cell = lambda v: f"{v:8.4f}" if v is not None else f"{'-':>8s}"
def table(title, key):
    L.append(f"\n{title}")
    L.append(f"{'':22s}" + "".join(f"{m:>8s}" for m in MODELS) + f"{'mean':>8s}")
    for tk in TECHS:
        v = [key(RUN[(m, tk)]) if (m, tk) in RUN else None for m in MODELS]
        ok = [x for x in v if x is not None]
        if ok:
            L.append(f"{tk:22s}" + "".join(cell(x) for x in v) + cell(np.mean(ok)))
    v = [np.mean([key(RUN[(m, tk)]) for tk in TECHS if (m, tk) in RUN]) if any((m, tk) in RUN for tk in TECHS) else None for m in MODELS]
    L.append(f"{'mean over techniques':22s}" + "".join(cell(x) for x in v))
table("TABLE 1a - TECHNIQUE x MODEL: dev macro-F1 (best global threshold)", f1)
table("TABLE 1b - TECHNIQUE x MODEL: dev macro-F1, per-statute thresholds, cross-fitted (honest)", cf)
table("TABLE 1c - TECHNIQUE x MODEL: dev micro-F1 (best global threshold)", lambda t: S[t]["dev"]["global"]["micro_f1"])

L.append("\nTABLE 2 - STUDY (LoRMA, same 12,000-document training subset): macro-F1 (global) | cross-fitted per-statute | "
         "micro-F1 | change vs reference (macro, global)")
for m in MODELS:
    st = sorted(t for t in FOUND if EXPS[t]["group"] == "S" and EXPS[t]["model"].upper() == m)
    if not st:
        continue
    ref = next((t for t in st if t.endswith("s_ref")), None)
    L.append(f"model {m}:")
    for t in st:
        d = S[t]["dev"]
        L.append(f"  {t:18s} {EXPS[t]['change'][:44]:44s} {f1(t):.4f} | {cf(t):.4f} | {d['global']['micro_f1']:.4f} | "
                 f"{(f1(t) - f1(ref)) if ref else 0:+.4f} | {S[t]['train_minutes']:5.1f} min | {S[t]['infer_ms_per_doc']:6.1f} ms/doc")

L.append("\nTABLE 3 - COST per technique (mean over models): trainable params M | % of model | peak train GB | train min | "
         "train tok/s | inference ms/doc | saved model MB")
for tk in TECHS:
    ts = [RUN[(m, tk)] for m in MODELS if (m, tk) in RUN]
    if ts:
        g = lambda k: np.mean([S[t].get(k) or 0 for t in ts])
        L.append(f"  {tk:9s} {g('params_trainable') / 1e6:7.2f}M | {100 * np.mean([S[t]['params_trainable'] / S[t]['params_total'] for t in ts]):5.1f}% | "
                 f"{g('peak_mem_gb'):4.1f} | {g('train_minutes'):5.1f} | {g('tok_s'):7,.0f} | {g('infer_ms_per_doc'):6.1f} | {g('model_mb'):7.1f}")
L.append("per model (LoRMA run): total params M | inference ms/doc | peak inference GB")
for m in MODELS:
    if (m, "lorma") in RUN:
        s = S[RUN[(m, "lorma")]]
        L.append(f"  {m}: {s['params_total'] / 1e6:7.2f}M | {s['infer_ms_per_doc']:6.1f} | {s.get('infer_mem_gb') or 0:4.2f}")

ranked = sorted(FOUND, key=f1, reverse=True)
top = [t for t in ranked if EXPS[t]["group"] != "S"][:8]
L.append("\nTABLE 4 - DECISION RULES (dev): macro-F1 / macro-P / macro-R / statutes per doc, for the best runs")
L.append(f"  {'run':22s} {'at 0.5':>26s} {'best global':>26s} {'per-statute (in-sample)':>26s}  cross-fitted: global / per-statute")
for t in top:
    d = S[t]["dev"]
    fmt = lambda r: f"{r['macro_f1']:.3f}/{r['macro_p']:.3f}/{r['macro_r']:.3f}/{r['pred_per_doc']:.2f}"
    L.append(f"  {t:22s} {fmt(d['at_0.5']):>26s} {fmt(d['global']):>26s} {fmt(d['per_label']):>26s}  "
             f"{d['crossfit']['global']:.4f} / {d['crossfit']['per_label']:.4f}")

L.append(f"\nTABLE 5 - RARE vs FREQUENT STATUTES (training frequency; dev macro-F1 per third, best global threshold)")
for t in top[:5]:
    b = E.frequency_buckets(S[t]["dev"]["global"]["per_label_f1"], freq)
    L.append(f"  {t:22s} " + "  ".join(f"freq {x['freq_min']}-{x['freq_max']}: {x['macro_f1']:.3f}" for x in b)
             + f"  | corr(F1, log freq) {np.corrcoef(S[t]['dev']['global']['per_label_f1'], np.log(freq))[0, 1]:+.2f}")

## Tables 6-9: ensembles, errors, co-occurrence, attention

In [3]:
# ---- ensembles (average of the statute probabilities), members chosen on dev
def avg(ts, split):
    src = DEV if split == "dev" else TEST
    return np.mean([src[t]["P"] for t in ts], axis=0)
tech_runs = [t for t in ranked if EXPS[t]["group"] != "S"]
best_of = lambda m: next((t for t in tech_runs if EXPS[t]["model"].upper() == m), None)
ENS = {"ens_top3": tech_runs[:3], "ens_top5": tech_runs[:5], "ens_top10": tech_runs[:10],
       "ens_bestpermodel": [b for b in (best_of(m) for m in MODELS) if b],
       "ens_lorma_all_models": [RUN[(m, "lorma")] for m in MODELS if (m, "lorma") in RUN]}
ENS = {k: v for k, v in ENS.items() if len(v) >= 2}
L.append("\nTABLE 6 - ENSEMBLES: macro-F1 at the best global threshold | threshold | per-statute in-sample (cross-fitted) | micro-F1 | members")
ENS_RES = {}
for k, ts in ENS.items():
    P = avg(ts, "dev")
    tg, fg = E.best_global(Y, P); tl = E.per_label_thresholds(Y, P, fallback=tg)
    c = E.crossfit_per_label(Y, P)
    ENS_RES[k] = dict(f1=fg, thr=tg, thr_l=tl, f1_l=E.macro_f1(Y, E.decide(P, tl)), cf=c["per_label"],
                      micro=E.report(Y, E.decide(P, tg))["micro_f1"])
    r = ENS_RES[k]
    L.append(f"  {k:22s} {fg:.4f} | {tg:.2f} | {r['f1_l']:.4f} ({r['cf']:.4f}) | {r['micro']:.4f} | {len(ts)}: {' '.join(ts)}")
best_ens = max(ENS_RES, key=lambda k: ENS_RES[k]["f1"]) if ENS_RES else None

# ---- error analysis of the best system (best ensemble if it beats the best single run)
if best_ens and ENS_RES[best_ens]["f1"] > f1(ranked[0]):
    name, P, thr = best_ens, avg(ENS[best_ens], "dev"), ENS_RES[best_ens]["thr"]
else:
    name, P, thr = ranked[0], DEV[ranked[0]]["P"], S[ranked[0]]["dev"]["global_threshold"]
pred = E.decide(P, thr)
r = E.report(Y, pred)
pl = np.array(r["per_label_f1"])
fn = ((Y == 1) & ~pred).sum(0); fp = ((Y == 0) & pred).sum(0)
L.append(f"\nTABLE 7 - ERRORS of {name} (threshold {thr:.2f}): {E.format_report(r)}")
L.append("  worst 10 statutes (F1 | dev support | training frequency | missed | wrongly added): " +
         "; ".join(f"{l}: {pl[l]:.2f} | {int(Y[:, l].sum())} | {int(freq[l])} | {int(fn[l])} | {int(fp[l])}" for l in np.argsort(pl)[:10]))
L.append("  best 10 statutes: " + "; ".join(f"{l}: {pl[l]:.2f} ({int(freq[l])})" for l in np.argsort(-pl)[:10]))
L.append("  most missed (FN): " + ", ".join(f"{l} ({int(fn[l])})" for l in np.argsort(-fn)[:10]))
L.append("  most over-predicted (FP): " + ", ".join(f"{l} ({int(fp[l])})" for l in np.argsort(-fp)[:10]))
# confused pairs: statute a is gold but missed while statute b is predicted but not gold, in the same document
miss, extra = ((Y == 1) & ~pred).astype(np.int32), ((Y == 0) & pred).astype(np.int32)
C = miss.T @ extra
np.fill_diagonal(C, 0)
pairs = np.dstack(np.unravel_index(np.argsort(-C, axis=None)[:12], C.shape))[0]
L.append("  most confused (gold missed -> predicted instead, documents): " + ", ".join(f"{a}->{b} ({int(C[a, b])})" for a, b in pairs))
ng = Y.sum(1)
for lo, hi in ((1, 1), (2, 3), (4, 6), (7, 99)):
    s = (ng >= lo) & (ng <= hi)
    if s.any():
        rr = E.report(Y[s], pred[s])
        L.append(f"  documents with {lo}-{hi} gold statutes (n {int(s.sum())}): micro-F1 {rr['micro_f1']:.4f}, subset acc "
                 f"{rr['subset_acc']:.4f}, predicted {rr['pred_per_doc']:.2f} vs gold {rr['gold_per_doc']:.2f}")
n_tok = DEV[ranked[0]]["n_tok"]
for lo, hi, lab in ((0, 512, "<= 512 tokens"), (512, 2048, "513-2,048"), (2048, 1e12, "> 2,048")):
    s = (n_tok > lo) & (n_tok <= hi)
    if s.any():
        L.append(f"  documents {lab} (n {int(s.sum())}): micro-F1 {E.report(Y[s], pred[s])['micro_f1']:.4f}")

# ---- statute co-occurrence (gold) and whether the co-occurrence layer helped
co = Y.T.astype(np.int32) @ Y.astype(np.int32)
np.fill_diagonal(co, 0)
top_pairs = np.dstack(np.unravel_index(np.argsort(-np.triu(co), axis=None)[:10], co.shape))[0]
L.append("\nTABLE 8 - STATUTE CO-OCCURRENCE (dev gold): most frequent pairs " +
         ", ".join(f"{a}+{b} ({int(co[a, b])})" for a, b in top_pairs) +
         f"; documents with >1 statute: {100 * (ng > 1).mean():.0f}%")
for m in MODELS:
    a, b = RUN.get((m, "s_ref")), RUN.get((m, "s_cooc"))
    if a and b:
        L.append(f"  model {m}: co-occurrence layer {f1(b):.4f} vs reference {f1(a):.4f} ({f1(b) - f1(a):+.4f}); "
                 f"subset acc {S[b]['dev']['global']['subset_acc']:.4f} vs {S[a]['dev']['global']['subset_acc']:.4f}")

# ---- where the attention goes (relative position of the chunk in the document)
L.append("\nTABLE 9 - WHERE THE ATTENTION GOES (dev documents read as several chunks): share of attention by relative position")
bins = np.linspace(0, 1, 6)
for t in [x for x in FOUND if x.endswith(("s_ref", "s_spread8", "s_lwan"))] + [RUN[(m, "lorma")] for m in "BD" if (m, "lorma") in RUN]:
    a, p = DEV[t]["att"], DEV[t]["pos"]
    multi = np.isfinite(a).sum(1) > 1
    ok = np.isfinite(a) & np.isfinite(p) & multi[:, None]
    if not ok.any():
        continue
    k = np.clip(np.digitize(p[ok], bins) - 1, 0, 4)
    share = [a[ok][k == b].sum() / a[ok].sum() for b in range(5)]
    nb = np.bincount(k, minlength=5) / ok.sum()
    L.append(f"  {t:20s} " + "  ".join(f"{int(100 * bins[b])}-{int(100 * bins[b + 1])}%: {100 * share[b]:4.1f} ({100 * nb[b]:4.1f})"
                                       for b in range(5)) + "   (attention share (share of chunks))")
print("\n".join(L))

=== REPORT phase 5 ===
data: {"train": {"rows_in_file": 42750, "unique_ids": 42750, "docs": 42750, "dropped_overlap_with_dev_test": 324, "tokens": 66616721, "tok_median": 592, "tok_p90": 3814, "tok_max": 124941, "labelled": true, "labels_per_doc": 3.699241031442983, "seconds": 94}, "dev": {"rows_in_file": 10181, "unique_ids": 10181, "docs": 10181, "tokens": 17845807, "tok_median": 744, "tok_p90": 4132, "tok_max": 106627, "labelled": true, "labels_per_doc": 3.9157253707887243, "seconds": 27}, "test": {"rows_in_file": 13019, "unique_ids": 13019, "docs": 13019, "tokens": 22251103, "tok_median": 736, "tok_p90": 3942, "tok_max": 208148, "labelled": false, "labels_per_doc": 0.0, "seconds": 33}, "n_labels": 100}
62 runs; 100 statutes; dev 10181 documents (3.92 statutes each); test 13019

ALL RUNS by dev macro-F1 at the best global threshold | threshold | per-statute thresholds: in-sample (cross-fitted) | at 0.5 | micro-F1 | subset acc | statutes/doc | trainable M | train min | peak GB | ms/do

## Submission files

In [4]:
# ---- submission files, each a different method; dev decides the order
N_RECOMMENDED = 10                       # Codabench allows 10 per day; files beyond this are backups
picks, seen = [], {}
def add(name, P_test, thr, why, dev_f1):
    pred = E.decide(P_test, thr)
    key = np.packbits(pred).tobytes()
    if key in seen:
        print(f"skip {name}: identical to {seen[key]}"); return
    seen[key] = name
    picks.append(dict(name=name, pred=pred, why=why, dev=dev_f1))
if not HAVE_TEST:
    print("no test probabilities in the bundles (test.json was missing) - no submission files")
else:
    # per-statute thresholds beat one global threshold on the honest (cross-fitted) dev estimate for every run, so
    # every file uses them; the dev column shows the cross-fitted macro-F1
    if best_ens:
        r = ENS_RES[best_ens]; Pt = avg(ENS[best_ens], "test")
        add(best_ens + "_perlabel", Pt, r["thr_l"], "best ensemble on dev, per-statute thresholds", r["cf"])
        add(best_ens, Pt, r["thr"], f"same, one global threshold {r['thr']:.2f}", r["f1"])
    use_pl = lambda t: S[t]["dev"]["crossfit"]["per_label"] > S[t]["dev"]["crossfit"]["global"]
    thr_of = lambda t: np.array(S[t]["dev"]["per_label_thresholds"]) if use_pl(t) else S[t]["dev"]["global_threshold"]
    f1 = lambda t: S[t]["dev"]["crossfit"]["per_label"] if use_pl(t) else S[t]["dev"]["global"]["macro_f1"]
    add(ranked[0] if EXPS[ranked[0]]["group"] != "S" else tech_runs[0], TEST[tech_runs[0]]["P"], thr_of(tech_runs[0]),
        "best single run on dev", f1(tech_runs[0]))
    tech_of = lambda t: t.split("_", 1)[1]
    best_tech = {tk: next((t for t in tech_runs if tech_of(t) == tk), None) for tk in TECHS}
    if best_tech["lorma"]:
        b = best_tech["lorma"]; add(b, TEST[b]["P"], thr_of(b), "best LoRMA run (compulsory technique)", f1(b))
    for m in MODELS:
        b = best_of(m)
        if b: add(b, TEST[b]["P"], thr_of(b), f"best run of model {m}", f1(b))
    covered = {tech_of(pk["name"]) for pk in picks if pk["name"] in EXPS}
    for tk in TECHS:
        b = best_tech[tk]
        if b and tk not in covered: add(b, TEST[b]["P"], thr_of(b), f"best {tk} run", f1(b))
    st = sorted([t for t in FOUND if EXPS[t]["group"] == "S"], key=f1, reverse=True)
    if st:
        add(st[0], TEST[st[0]]["P"], thr_of(st[0]), f"best study variant ({EXPS[st[0]]['change']})", f1(st[0]))
    for k in sorted(ENS_RES, key=lambda k: -ENS_RES[k]["cf"]):
        add(k + "_perlabel", avg(ENS[k], "test"), ENS_RES[k]["thr_l"], "ensemble, per-statute thresholds", ENS_RES[k]["cf"])
    SUB = f"{W}/submissions"; shutil.rmtree(SUB, ignore_errors=True); os.makedirs(SUB)
    rows = []
    for n, pk in enumerate(picks, 1):
        fn_ = f"{n:02d}_{pk['name']}.zip"
        pj = f"{W}/_prediction.json"
        D.write_prediction(test_ids, [np.flatnonzero(r).tolist() for r in pk["pred"]], pj, K)
        with zipfile.ZipFile(f"{SUB}/{fn_}", "w", zipfile.ZIP_DEFLATED) as z:
            z.write(pj, arcname="prediction.json")
        rows.append(f"{fn_},{pk['dev']:.4f},{pk['pred'].sum(1).mean():.2f},{len(test_ids)},\"{pk['why']}\"")
    open(f"{W}/SUBMISSIONS.csv", "w").write("file,dev_macro_f1_honest,test_statutes_per_doc,test_docs,why\n" + "\n".join(rows) + "\n")
    with zipfile.ZipFile(f"{W}/all_submissions.zip", "w") as z:
        for f in sorted(os.listdir(SUB)):
            z.write(f"{SUB}/{f}", arcname=f)
        z.write(f"{W}/SUBMISSIONS.csv", arcname="SUBMISSIONS.csv")
    L.append(f"\nSUBMISSIONS (submit in this order; the first {N_RECOMMENDED} are recommended, the rest are backups)")
    L += ["  " + r for r in rows]
    print("\n".join(L[-len(rows) - 1:]))
open(f"{W}/REPORT_p5_final.txt", "w").write("\n".join(L))
print("\n=== END REPORT ===")

skip D07_full: identical to D07_full
skip ens_top5_perlabel: identical to ens_top5_perlabel

SUBMISSIONS (submit in this order; the first 10 are recommended, the rest are backups)
  01_ens_top5_perlabel.zip,0.3864,4.85,13019,"best ensemble on dev, per-statute thresholds"
  02_ens_top5.zip,0.3650,5.93,13019,"same, one global threshold 0.15"
  03_D07_full.zip,0.3805,4.86,13019,"best single run on dev"
  04_D01_lorma.zip,0.3655,5.04,13019,"best LoRMA run (compulsory technique)"
  05_A07_full.zip,0.3255,5.74,13019,"best run of model A"
  06_B07_full.zip,0.3780,4.94,13019,"best run of model B"
  07_C07_full.zip,0.3744,5.02,13019,"best run of model C"
  08_E07_full.zip,0.3705,4.91,13019,"best run of model E"
  09_D02_adalora.zip,0.3596,5.05,13019,"best adalora run"
  10_D03_qlora.zip,0.3759,5.17,13019,"best qlora run"
  11_D04_adapter.zip,0.3501,5.33,13019,"best adapter run"
  12_B05_prefix.zip,0.3372,5.28,13019,"best prefix run"
  13_D06_partial.zip,0.3745,4.98,13019,"best partial run"
  14